In [1]:
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 74.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 126.6 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [scikit-learn] [scikit-learn]


In [2]:
# ============================================================================
# MELD Multimodal Emotion Recognition — Kaggle Benchmark Task
# ============================================================================
# Task: Given a video clip + dialogue transcript from the MELD dataset,
#        predict the emotion of the target utterance.
# Input: Video (.mp4) + Text (utterance + dialogue context)
# Output: One of 7 emotion labels
# Metric: Weighted-Average F1 Score (float, 0.0 – 1.0)
# Samples: 20 (stratified across emotion classes)
# ============================================================================

import kaggle_benchmarks as kbench
from kaggle_benchmarks.content_types import images
from dataclasses import dataclass
from sklearn.metrics import f1_score
import pandas as pd
import base64
import os
import re

# ============================================================================
# CONFIGURATION
# ============================================================================

DATASET_ROOT = "/kaggle/input/datasets/zaber666/meld-dataset"
NUM_SAMPLES = 50

# Fixed random seed — ensures every model gets evaluated on the exact same samples.
# This is critical: Kaggle re-runs this notebook per model, only swapping kbench.llm.
# A fixed seed guarantees reproducible, identical sample selection across all runs.
RANDOM_SEED = 42

VALID_EMOTIONS = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]

# Mapping of common LLM synonyms → canonical MELD labels
EMOTION_SYNONYMS = {
    "happy": "joy",
    "happiness": "joy",
    "excited": "joy",
    "sad": "sadness",
    "angry": "anger",
    "mad": "anger",
    "furious": "anger",
    "scared": "fear",
    "afraid": "fear",
    "fearful": "fear",
    "disgusted": "disgust",
    "disgusting": "disgust",
    "surprised": "surprise",
    "shocking": "surprise",
    "shocked": "surprise",
}

# Number of preceding utterances to include as dialogue context
CONTEXT_WINDOW = 3

# Maximum video file size in bytes (~0.5MB ≈ ~2.7M base64 chars ≈ ~700K tokens)
# Gemini models have 1M token context; we keep video under ~500K tokens to leave
# room for the prompt text + response. 1.5MB raw ≈ 2MB base64 ≈ ~500K tokens.
MAX_VIDEO_FILE_SIZE_BYTES = 0.1 * 1024 * 1024  

# ============================================================================
# STRUCTURED OUTPUT SCHEMA
# ============================================================================

@dataclass
class EmotionPrediction:
    """The LLM must return exactly one emotion label."""
    emotion: str


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def find_csv_and_video_dir():
    """
    Locate the test CSV and video directory within the dataset.
    Handles the MELD dataset's nested folder structure.
    Falls back to dev split if test videos are unusable (macOS ._ artifacts).
    """
    # --- Try test split first ---
    test_csv_candidates = [
        os.path.join(DATASET_ROOT, "MELD.Raw", "test_sent_emo.csv"),
        os.path.join(DATASET_ROOT, "MELD-RAW", "MELD.Raw", "test_sent_emo.csv"),
    ]
    test_video_candidates = [
        os.path.join(DATASET_ROOT, "MELD.Raw", "test", "output_repeated_splits_test"),
        os.path.join(DATASET_ROOT, "MELD-RAW", "MELD.Raw", "test", "output_repeated_splits_test"),
    ]

    # --- Also prepare dev split as fallback ---
    dev_csv_candidates = [
        os.path.join(DATASET_ROOT, "MELD.Raw", "dev_sent_emo.csv"),
        os.path.join(DATASET_ROOT, "MELD-RAW", "MELD.Raw", "dev_sent_emo.csv"),
    ]
    dev_video_candidates = [
        os.path.join(DATASET_ROOT, "MELD.Raw", "dev", "dev_splits_complete"),
        os.path.join(DATASET_ROOT, "MELD-RAW", "MELD.Raw", "dev", "dev_splits_complete"),
    ]

    # Try test split
    csv_path = _find_existing(test_csv_candidates)
    video_dir = _find_existing(test_video_candidates)

    if csv_path and video_dir:
        # Check if test videos are real files (not just macOS ._ artifacts)
        real_videos = [f for f in os.listdir(video_dir)
                       if f.endswith(".mp4") and not f.startswith("._")]
        if len(real_videos) > 0:
            return csv_path, video_dir, "test"

    # Fallback to dev split
    csv_path = _find_existing(dev_csv_candidates)
    video_dir = _find_existing(dev_video_candidates)

    if csv_path and video_dir:
        return csv_path, video_dir, "dev"

    # Last resort: walk the directory tree to find any CSV + video folder
    csv_path, video_dir = _walk_find(DATASET_ROOT)
    return csv_path, video_dir, "discovered"


def _find_existing(candidates):
    """Return the first path that exists, or None."""
    for p in candidates:
        if os.path.exists(p):
            return p
    return None


def _walk_find(root):
    """Walk directory tree looking for a *_sent_emo.csv and a folder with .mp4 files."""
    found_csv = None
    found_video_dir = None
    for dirpath, dirnames, filenames in os.walk(root):
        for f in filenames:
            if f.endswith("_sent_emo.csv") and found_csv is None:
                found_csv = os.path.join(dirpath, f)
        mp4s = [f for f in filenames if f.endswith(".mp4") and not f.startswith("._")]
        if len(mp4s) > 10 and found_video_dir is None:
            found_video_dir = dirpath
    return found_csv, found_video_dir


def load_and_prepare_data(csv_path):
    """
    Load the CSV, normalize columns, and sort utterances within each dialogue.
    """
    df = pd.read_csv(csv_path)

    # Normalize column names (MELD CSVs can have inconsistent casing/spacing)
    df.columns = df.columns.str.strip().str.lower()

    # Ensure required columns exist
    col_map = {}
    for col in df.columns:
        cl = col.lower().strip()
        if "utterance" in cl and "id" in cl:
            col_map["utterance_id"] = col
        elif "dialogue" in cl and "id" in cl:
            col_map["dialogue_id"] = col
        elif "emotion" == cl:
            col_map["emotion"] = col
        elif "utterance" == cl:
            col_map["utterance"] = col
        elif "speaker" == cl:
            col_map["speaker"] = col
        elif "sentiment" == cl:
            col_map["sentiment"] = col

    # Sort by dialogue, then utterance order within each dialogue
    df = df.sort_values([col_map["dialogue_id"], col_map["utterance_id"]]).reset_index(drop=True)

    return df, col_map


def select_stratified_samples(df, col_map, video_dir, n=NUM_SAMPLES):
    """
    Select N samples via stratified sampling, ensuring each has a valid video file
    that fits within the context window.

    Deterministic: uses RANDOM_SEED so every model run gets the exact same samples.

    Strategy:
    1. Filter to utterances with valid, small-enough video files.
    2. Stratified sample proportional to class distribution (preserves MELD's
       natural imbalance, which is important for Weighted-Average F1).
    3. Guarantee minimum 2 samples per emotion class (so even fear/disgust
       contribute to per-class F1).
    """
    emotion_col = col_map["emotion"]
    dia_col = col_map["dialogue_id"]
    utt_col = col_map["utterance_id"]

    # Normalize emotion labels in the dataframe
    df["_emotion_lower"] = df[emotion_col].str.strip().str.lower()

    # ------------------------------------------------------------------
    # Step 1: Filter to valid samples (video exists + fits in context)
    # ------------------------------------------------------------------
    valid_indices = []
    skipped_too_large = 0
    skipped_missing = 0
    for idx, row in df.iterrows():
        video_fname = f"dia{row[dia_col]}_utt{row[utt_col]}.mp4"
        video_path = os.path.join(video_dir, video_fname)
        if os.path.isfile(video_path):
            fsize = os.path.getsize(video_path)
            if fsize <= MAX_VIDEO_FILE_SIZE_BYTES:
                valid_indices.append(idx)
            else:
                skipped_too_large += 1
        else:
            skipped_missing += 1

    valid_df = df.loc[valid_indices].copy()
    print(f"[INFO] Found {len(valid_df)} utterances with valid video files out of {len(df)} total.")
    print(f"[INFO] Skipped {skipped_too_large} videos exceeding "
          f"{MAX_VIDEO_FILE_SIZE_BYTES/1024/1024:.1f}MB size limit.")
    print(f"[INFO] Skipped {skipped_missing} utterances with missing video files.")

    if len(valid_df) == 0:
        raise FileNotFoundError(
            f"No valid video files found in {video_dir}. "
            f"Checked pattern: dia<ID>_utt<ID>.mp4"
        )

    # ------------------------------------------------------------------
    # Step 2: Stratified sampling with guaranteed minimums per class
    # ------------------------------------------------------------------
    min_per_class = 2
    actual_n = min(n, len(valid_df))  # Can't sample more than available

    # Count available samples per emotion
    class_counts = valid_df["_emotion_lower"].value_counts()
    present_emotions = [e for e in VALID_EMOTIONS if e in class_counts.index]
    print(f"[INFO] Available emotion distribution in valid pool:")
    for emo in present_emotions:
        print(f"  {emo}: {class_counts[emo]}")

    # First pass: guarantee minimum samples per class
    guaranteed = []
    remaining_budget = actual_n

    for emo in present_emotions:
        emo_pool = valid_df[valid_df["_emotion_lower"] == emo]
        take = min(min_per_class, len(emo_pool))
        sampled = emo_pool.sample(n=take, random_state=RANDOM_SEED)
        guaranteed.append(sampled)
        remaining_budget -= take

    guaranteed_df = pd.concat(guaranteed)
    guaranteed_indices = set(guaranteed_df.index)

    # Second pass: fill remaining budget proportionally from leftover pool
    leftover_pool = valid_df[~valid_df.index.isin(guaranteed_indices)]

    if remaining_budget > 0 and len(leftover_pool) > 0:
        take_more = min(remaining_budget, len(leftover_pool))

        # Proportional stratified sampling from the leftover
        proportional_parts = []
        for emo, grp in leftover_pool.groupby("_emotion_lower"):
            take = min(
                len(grp),
                max(1, int(round(take_more * len(grp) / len(leftover_pool))))
            )
            proportional_parts.append(grp.sample(n=take, random_state=RANDOM_SEED))
        proportional = pd.concat(proportional_parts)

        # If proportional allocation gave us too many or too few, adjust
        if len(proportional) > take_more:
            proportional = proportional.sample(n=take_more, random_state=RANDOM_SEED)
        elif len(proportional) < take_more:
            still_unused = leftover_pool[~leftover_pool.index.isin(proportional.index)]
            extra = min(take_more - len(proportional), len(still_unused))
            if extra > 0:
                proportional = pd.concat([
                    proportional,
                    still_unused.sample(n=extra, random_state=RANDOM_SEED)
                ])

        selected_df = pd.concat([guaranteed_df, proportional])
    else:
        selected_df = guaranteed_df

    # Final sort for reproducible ordering across runs
    selected_df = selected_df.sort_values(
        [col_map["dialogue_id"], col_map["utterance_id"]]
    ).reset_index(drop=True)

    print(f"\n[INFO] Final selection: {len(selected_df)} samples across emotions:")
    print(selected_df["_emotion_lower"].value_counts().to_string())
    return selected_df


def build_dialogue_context(df, col_map, target_row, n_context=CONTEXT_WINDOW):
    """
    Build the dialogue context string: up to n_context preceding utterances
    from the same dialogue, with speaker labels.
    """
    dia_col = col_map["dialogue_id"]
    utt_col = col_map["utterance_id"]
    utt_text_col = col_map["utterance"]
    speaker_col = col_map["speaker"]

    same_dialogue = df[df[dia_col] == target_row[dia_col]].sort_values(utt_col)
    target_utt_id = target_row[utt_col]

    preceding = same_dialogue[same_dialogue[utt_col] < target_utt_id].tail(n_context)

    context_lines = []
    for _, row in preceding.iterrows():
        speaker = str(row[speaker_col]).strip()
        text = str(row[utt_text_col]).strip()
        context_lines.append(f"[{speaker}]: {text}")

    return "\n".join(context_lines)


def build_prompt(dialogue_context, speaker, utterance):
    """
    Construct the full text prompt for the LLM.
    """
    prompt = (
        "You are an expert at recognizing emotions in conversations. "
        "You will be shown a short video clip from a TV show along with "
        "the dialogue transcript. Your task is to identify the emotion "
        "expressed in the TARGET utterance.\n\n"
    )

    if dialogue_context:
        prompt += "PRECEDING DIALOGUE CONTEXT:\n"
        prompt += dialogue_context + "\n\n"

    prompt += "TARGET UTTERANCE TO CLASSIFY:\n"
    prompt += f"[{speaker}]: {utterance}\n\n"

    prompt += (
        "Based on BOTH the video (facial expressions, vocal tone, body language) "
        "and the text above, classify the emotion of the TARGET utterance as "
        "exactly one of the following:\n"
        "anger, disgust, fear, joy, neutral, sadness, surprise\n\n"
        "Respond with a single JSON object containing one field 'emotion' "
        "with your chosen label."
    )
    return prompt


def load_video_as_base64(video_path):
    """
    Read a video file and encode it as base64 for multimodal LLM input.
    Returns a base64 string.
    """
    with open(video_path, "rb") as f:
        video_bytes = f.read()
    return base64.b64encode(video_bytes).decode("utf-8")


def normalize_emotion(raw_prediction):
    """
    Normalize the LLM's predicted emotion to a canonical MELD label.
    Handles synonyms, casing, and extra whitespace.
    Returns the canonical label or 'INVALID' if unrecognizable.
    """
    cleaned = raw_prediction.strip().lower()

    # Remove any surrounding quotes or punctuation
    cleaned = re.sub(r"[^a-z]", "", cleaned)

    if cleaned in VALID_EMOTIONS:
        return cleaned

    if cleaned in EMOTION_SYNONYMS:
        return EMOTION_SYNONYMS[cleaned]

    # Fuzzy last resort: check if any valid emotion is a substring
    for emo in VALID_EMOTIONS:
        if emo in cleaned:
            return emo

    return "INVALID"


# ============================================================================
# MAIN BENCHMARK TASK
# ============================================================================

@kbench.task(
    name="meld_multimodal_emotion_recognition",
    description=(
        "Multimodal emotion recognition on the MELD dataset. "
        "The model receives a video clip (.mp4) and dialogue transcript, "
        "then predicts one of 7 emotions: anger, disgust, fear, joy, "
        "neutral, sadness, surprise. Scored by Weighted-Average F1 (0–1)."
    ),
)
def meld_multimodal_emotion_recognition(llm) -> float:
    """
    Evaluate an LLM's ability to recognize emotions from video + text.
    Returns Weighted-Average F1 score between 0.0 and 1.0.
    """
    # ------------------------------------------------------------------
    # STEP 1: Locate data files
    # ------------------------------------------------------------------
    csv_path, video_dir, split_name = find_csv_and_video_dir()
    print(f"[INFO] Using split: {split_name}")
    print(f"[INFO] CSV: {csv_path}")
    print(f"[INFO] Video dir: {video_dir}")

    # ------------------------------------------------------------------
    # STEP 2: Load and prepare the data
    # ------------------------------------------------------------------
    df, col_map = load_and_prepare_data(csv_path)
    print(f"[INFO] Loaded {len(df)} utterances from CSV.")

    # ------------------------------------------------------------------
    # STEP 3: Select 20 stratified samples
    # ------------------------------------------------------------------
    samples = select_stratified_samples(df, col_map, video_dir, n=NUM_SAMPLES)

    # ------------------------------------------------------------------
    # STEP 4: Run predictions
    # ------------------------------------------------------------------
    y_true = []
    y_pred = []
    results_log = []

    dia_col = col_map["dialogue_id"]
    utt_col = col_map["utterance_id"]
    utt_text_col = col_map["utterance"]
    speaker_col = col_map["speaker"]

    for i, (idx, row) in enumerate(samples.iterrows()):
        ground_truth = row["_emotion_lower"]
        speaker = str(row[speaker_col]).strip()
        utterance = str(row[utt_text_col]).strip()
        video_fname = f"dia{row[dia_col]}_utt{row[utt_col]}.mp4"
        video_path = os.path.join(video_dir, video_fname)

        # Build dialogue context from preceding utterances
        context = build_dialogue_context(df, col_map, row, n_context=CONTEXT_WINDOW)

        # Build the text prompt
        prompt_text = build_prompt(context, speaker, utterance)

        # Load video as base64
        video_b64 = load_video_as_base64(video_path)

        print(f"\n[SAMPLE {i+1}/{NUM_SAMPLES}] dia{row[dia_col]}_utt{row[utt_col]} | "
              f"Ground truth: {ground_truth}")

        # ----------------------------------------------------------
        # Prompt the LLM with video + text
        # ----------------------------------------------------------
        # Fresh chat context per sample so history doesn't leak
        with kbench.chats.new(f"sample_{i}"):
            try:
                # Primary: video + text with structured output
                prediction = llm.prompt(
                    [
                        {
                            "type": "video",
                            "source": {
                                "type": "base64",
                                "media_type": "video/mp4",
                                "data": video_b64,
                            },
                        },
                        {"type": "text", "text": prompt_text},
                    ],
                    schema=EmotionPrediction,
                )
                raw_emotion = prediction.emotion

            except Exception as e_schema:
                print(f"  [WARN] Video+schema failed ({type(e_schema).__name__}): {e_schema}")
                print(f"  [FALLBACK] Trying video + text without schema...")

                # Fallback: video + text without schema (still multimodal)
                # Must use a fresh chat so the failed attempt doesn't pollute context
                with kbench.chats.new(f"sample_{i}_retry"):
                    raw_response = llm.prompt(
                        [
                            {
                                "type": "video",
                                "source": {
                                    "type": "base64",
                                    "media_type": "video/mp4",
                                    "data": video_b64,
                                },
                            },
                            {"type": "text", "text": prompt_text},
                        ]
                    )
                    raw_emotion = raw_response.strip()

        # Normalize the prediction
        predicted_emotion = normalize_emotion(raw_emotion)

        print(f"  Raw LLM output: '{raw_emotion}' → Normalized: '{predicted_emotion}'")
        print(f"  {'✓ CORRECT' if predicted_emotion == ground_truth else '✗ WRONG'}")

        y_true.append(ground_truth)
        y_pred.append(predicted_emotion)
        results_log.append({
            "sample": i + 1,
            "dialogue_id": row[dia_col],
            "utterance_id": row[utt_col],
            "speaker": speaker,
            "utterance": utterance[:60] + "..." if len(utterance) > 60 else utterance,
            "ground_truth": ground_truth,
            "predicted": predicted_emotion,
            "raw_output": raw_emotion[:80] if isinstance(raw_emotion, str) else str(raw_emotion)[:80],
            "correct": predicted_emotion == ground_truth,
        })

    # ------------------------------------------------------------------
    # STEP 5: Compute Weighted-Average F1 Score
    # ------------------------------------------------------------------
    # Replace INVALID predictions with a dummy label for F1 computation
    # (they'll get 0 credit since they won't match any ground truth)
    y_pred_clean = [p if p != "INVALID" else "INVALID" for p in y_pred]

    # Include all possible labels to ensure consistent computation
    all_labels = VALID_EMOTIONS + (["INVALID"] if "INVALID" in y_pred_clean else [])

    waf1 = f1_score(
        y_true,
        y_pred_clean,
        average="weighted",
        labels=all_labels,
        zero_division=0,
    )

    # ------------------------------------------------------------------
    # STEP 6: Print summary
    # ------------------------------------------------------------------
    print("\n" + "=" * 70)
    print("RESULTS SUMMARY")
    print("=" * 70)

    results_df = pd.DataFrame(results_log)
    print(results_df[["sample", "ground_truth", "predicted", "correct"]].to_string(index=False))

    correct_count = sum(r["correct"] for r in results_log)
    invalid_count = sum(1 for p in y_pred if p == "INVALID")

    print(f"\nAccuracy:  {correct_count}/{NUM_SAMPLES} ({correct_count/NUM_SAMPLES:.1%})")
    print(f"Invalid:   {invalid_count}/{NUM_SAMPLES}")
    print(f"WAF1:      {waf1:.4f}")
    print("=" * 70)

    # Assert the score is in valid range
    kbench.assertions.assert_true(
        0.0 <= waf1 <= 1.0,
        expectation="Weighted-Average F1 score should be between 0 and 1."
    )

    return waf1


# ============================================================================
# RUN THE TASK
# ============================================================================

# Uses kbench.llm placeholder — Kaggle swaps in the actual model
# when you click "Add Models" on the Task Detail page.
meld_multimodal_emotion_recognition.run(kbench.llm)

[INFO] Using split: test
[INFO] CSV: /kaggle/input/datasets/zaber666/meld-dataset/MELD-RAW/MELD.Raw/test_sent_emo.csv
[INFO] Video dir: /kaggle/input/datasets/zaber666/meld-dataset/MELD-RAW/MELD.Raw/test/output_repeated_splits_test
[INFO] Loaded 2610 utterances from CSV.


[INFO] Found 65 utterances with valid video files out of 2610 total.
[INFO] Skipped 2545 videos exceeding 0.1MB size limit.
[INFO] Skipped 0 utterances with missing video files.
[INFO] Available emotion distribution in valid pool:
  anger: 13
  joy: 11
  neutral: 29
  sadness: 4
  surprise: 8

[INFO] Final selection: 65 samples across emotions:
_emotion_lower
neutral     29
anger       13
joy         11
surprise     8
sadness      4

[SAMPLE 1/250] dia8_utt3 | Ground truth: surprise


  Raw LLM output: 'surprise' → Normalized: 'surprise'
  ✓ CORRECT

[SAMPLE 2/250] dia14_utt3 | Ground truth: joy


  Raw LLM output: 'joy' → Normalized: 'joy'
  ✓ CORRECT

[SAMPLE 3/250] dia25_utt4 | Ground truth: anger


KeyboardInterrupt: 